In [31]:
import pathlib
import sys

import duckdb
import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display

from irp.anomalies.blacklist import add as _bl_add
from irp.anomalies.checklist import add as _chk_add
from irp.anomalies.whitelist import add as _wl_add

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()


def _html(df):
    return HTML(
        '<style>table td, table th { white-space: nowrap; }</style>'
        '<div style="overflow-x:auto;overflow-y:auto;max-height:500px;'
        'width:100%;display:block;border:1px solid #444">'
        + df.to_html(index=False, escape=False)
        + '</div>'
    )


def _fmt_value(v):
    if pd.isna(v): return ''
    if abs(v) >= 1000: return f'{v:,.0f}'
    if abs(v) >= 1:    return f'{v:,.2f}'
    return f'{v:.4f}'

def _prep_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'rule' in df.columns:
        df = df.drop(columns=['rule'])
    if 'company_name' in df.columns:
        loc = df.columns.get_loc('company_name')
        if isinstance(loc, int):
            df.insert(loc, 'company', df['company_name'].str[:10])
        df = df.drop(columns=['company_name'])
    if 'edgar_url' in df.columns:
        df['x'] = df['edgar_url'].apply(
            lambda u: (
                f'<a href="{u}" target="_blank">xxxxx</a>' if pd.notna(u) and u else ''
            )
        )
        df = df.drop(columns=['edgar_url'])
    if 'value' in df.columns:
        df['value'] = df['value'].apply(_fmt_value)
    return df


def _exc_widgets(get_row_fn):
    row_in = w.BoundedIntText(
        min=0, max=0, value=0, description='Row #:', layout=w.Layout(width='150px')
    )
    note_in = w.Text(
        placeholder='Comment (optional)',
        description='Note:',
        layout=w.Layout(width='380px'),
    )
    wl_btn = w.Button(description='Whitelist', button_style='success')
    bl_btn = w.Button(description='Flag error', button_style='danger')
    chk_btn = w.Button(description='Check', button_style='warning')
    msg_out = w.Output()

    def _make_handler(add_fn):
        def _add(_):
            msg_out.clear_output(wait=True)
            row = get_row_fn(row_in.value)
            if row is None:
                with msg_out:
                    print('No findings.')
                return
            added = add_fn(
                str(row['ticker']),
                str(row['period']),
                str(row['rule']),
                str(row['column']),
                _fmt_value(row['value']),
                note_in.value.strip(),
            )
            with msg_out:
                label = (
                    f"{row['ticker']} {row['period']} / {row['rule']} / {row['column']}"
                )
                print(f'Added: {label}' if added else 'Already exists.')

        return _add

    wl_btn.on_click(_make_handler(_wl_add))
    bl_btn.on_click(_make_handler(_bl_add))
    chk_btn.on_click(_make_handler(_chk_add))
    return row_in, note_in, wl_btn, chk_btn, bl_btn, msg_out


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


## 1. Load Data

In [32]:
TABLE_NAMES = (
    'income',
    'balance',
    'cashflow',
    'companies',
    'industries',
    'sec_filings',
)
data = {name: q(f'SELECT * FROM "{name}"') for name in TABLE_NAMES}
for name, df in data.items():
    print(f'  {name:<15} {len(df):>12,} rows')

  income                68,495 rows
  balance               68,486 rows
  cashflow              68,486 rows
  companies              6,556 rows
  industries                74 rows
  sec_filings           68,328 rows


In [33]:
# Tickers absent from SEC EDGAR (foreign/OTC/delisted — no 10-K/10-Q filings)
no_edgar = set(
    data['sec_filings']
    .loc[data['sec_filings']['error'] == 'ticker not found', 'ticker']
    .unique()
)
print(f'{len(no_edgar):,} tickers not found in SEC EDGAR')

EXCLUDE_NO_EDGAR = True  # set False to include all tickers

if EXCLUDE_NO_EDGAR:
    for name in ('income', 'balance', 'cashflow'):
        before = len(data[name])
        data[name] = data[name][~data[name]['Ticker'].isin(no_edgar)].reset_index(
            drop=True
        )
        print(
            f'  {name}: {before:,} -> {len(data[name]):,} rows ({before - len(data[name]):,} dropped)'
        )

1,519 tickers not found in SEC EDGAR
  income: 68,495 -> 50,828 rows (17,667 dropped)
  balance: 68,486 -> 50,822 rows (17,664 dropped)
  cashflow: 68,486 -> 50,822 rows (17,664 dropped)


## 2. Run Quality Rules

In [34]:
from irp.anomalies import run
from irp.anomalies.blacklist import load as load_blacklist
from irp.anomalies.blacklist import suppress as suppress_bl
from irp.anomalies.checklist import load as load_checklist
from irp.anomalies.checklist import suppress as suppress_chk
from irp.anomalies.whitelist import load as load_whitelist
from irp.anomalies.whitelist import suppress as suppress_wl

findings = run(data)
_wl = load_whitelist()
_bl = load_blacklist()
_chk = load_checklist()
print(f'{len(_wl)} whitelisted, {len(_bl)} blacklisted, {len(_chk)} on checklist')
findings = suppress_wl(findings)
findings = suppress_bl(findings)
findings = suppress_chk(findings)
print(f'Total findings after suppression: {len(findings):,}')

14 whitelisted, 39 blacklisted, 0 on checklist
Total findings after suppression: 24,856


## 3. Summary

In [35]:
summary = (
    findings.groupby('rule')['ticker']
    .count()
    .rename('count')
    .reset_index()
    .sort_values('count', ascending=False)
)
print(f'Total findings: {len(findings):,}')
display(_html(summary))

Total findings: 24,856


rule,count
sector_outlier,15262
sudden_jump,9428
accounting_identity,156
impossible_value,10


## 4. Impossible Values

Checks: negative Revenue, non-positive Shares (Diluted/Basic), non-positive Total Assets.

In [36]:
impos = (
    findings[findings['rule'] == 'impossible_value'][
        [
            'ticker',
            'company_name',
            'period',
            'report_date',
            'column',
            'value',
            'rule',
            'detail',
            'edgar_url',
        ]
    ]
    .sort_values(['ticker', 'column', 'period'])
    .reset_index(drop=True)
)
print(f'{len(impos)} violations')
_impos_show = _prep_df(impos.copy())
_impos_show.insert(0, '#', range(1, len(_impos_show) + 1))
display(_html(_impos_show))

_impos_row, _impos_note, _impos_wl_btn, _impos_chk_btn, _impos_bl_btn, _impos_msg = (
    _exc_widgets(lambda i: impos.iloc[i - 1] if 0 < i <= len(impos) else None)
)
_impos_row.max = max(len(impos), 1)
display(
    w.HBox([_impos_row, _impos_note, _impos_wl_btn, _impos_chk_btn, _impos_bl_btn]),
    _impos_msg,
)

10 violations


#,ticker,company,period,report_date,column,value,detail,x
1,TLSA,Tiziana Li,2021FY,2021-12-31,Revenue,"-855,000",Revenue is negative,xxxxx
2,TNGX,Tango Ther,2020Q3,2020-09-30,Revenue,"-11,261,000",Revenue is negative,xxxxx
3,TTE,TotalEnerg,2021FY,2021-12-31,Revenue,"-139,851,000,000",Revenue is negative,xxxxx
4,TTE,TotalEnerg,2022FY,2022-12-31,Revenue,"-187,137,000,000",Revenue is negative,xxxxx
5,TXMD,Therapeuti,2023Q3,2023-09-30,Revenue,"-53,000",Revenue is negative,xxxxx
6,VIR,Vir Biotec,2022Q2,2022-06-30,Revenue,"-40,629,000",Revenue is negative,xxxxx
7,VRCA,Verrica Ph,2024Q3,2024-09-30,Revenue,"-1,781,000",Revenue is negative,xxxxx
8,WDH,Waterdrop,2021FY,2021-12-31,Revenue,"-38,519,331",Revenue is negative,xxxxx
9,WKHS,Workhorse,2021FY,2021-12-31,Revenue,"-851,922",Revenue is negative,xxxxx
10,WKHS,Workhorse,2021Q3,2021-09-30,Revenue,"-576,602",Revenue is negative,xxxxx


Output()

## 5. Accounting Identity Violations

Balance sheet check:  within 1% tolerance.

In [7]:
acct = (
    findings[findings['rule'] == 'accounting_identity'][
        ['ticker', 'company_name', 'period', 'report_date', 'value', 'rule', 'detail', 'edgar_url']
    ]
    .sort_values('value', ascending=False)
    .reset_index(drop=True)
)
print(f'{len(acct)} violations')
_acct_show = _prep_df(acct.copy())
_acct_show.insert(0, '#', range(1, len(_acct_show) + 1))
display(_html(_acct_show))

_acct_row, _acct_note, _acct_wl_btn, _acct_chk_btn, _acct_bl_btn, _acct_msg = (
    _exc_widgets(lambda i: acct.iloc[i - 1] if 0 < i <= len(acct) else None)
)
_acct_row.max = max(len(acct), 1)
display(
    w.HBox([_acct_row, _acct_note, _acct_wl_btn, _acct_chk_btn, _acct_bl_btn]),
    _acct_msg,
)

156 violations


#,ticker,company,period,report_date,value,detail,x
1,LODE,Comstock I,2022FY,2022-12-31,"2,218","Annual=178150, Q1+Q2+Q3+Q4=395326225, rel_err=221806.4%",xxxxx
2,LODE,Comstock I,2021FY,2021-12-31,"2,020","Annual=862165, Q1+Q2+Q3+Q4=1742521540, rel_err=202010.0%",xxxxx
3,AQMS,Aqua Metal,2023FY,2023-12-31,101.84,"Annual=25000, Q1+Q2+Q3+Q4=2571000, rel_err=10184.0%",xxxxx
4,OMEX,Odyssey Ma,2021FY,2021-12-31,4.09,"Assets=8908887, Liab+Eq=45327595, rel_err=408.79%",xxxxx
5,MIR,Mirion Tec,2021FY,2021-12-31,3.34,"Annual=154100000, Q1+Q2+Q3+Q4=668300000, rel_err=333.7%",xxxxx
6,OMEX,Odyssey Ma,2022FY,2022-12-31,3.12,"Assets=13870827, Liab+Eq=57200799, rel_err=312.38%",xxxxx
7,OMEX,Odyssey Ma,2020FY,2020-12-31,2.57,"Assets=11759464, Liab+Eq=41986854, rel_err=257.05%",xxxxx
8,OMEX,Odyssey Ma,2023FY,2023-12-31,2.35,"Assets=22752297, Liab+Eq=76253552, rel_err=235.15%",xxxxx
9,WOR,WORTHINGTO,2023FY,2023-05-31,1.33,"Annual=1418496000, Q1+Q2+Q3+Q4=3299323000, rel_err=132.6%",xxxxx
10,VTSI,"VirTra, In",2024FY,2024-12-31,1.00,"Annual=747977, Q1+Q2+Q3+Q4=0, rel_err=100.0%",xxxxx


Output()

## 6. Sector Outliers

IQR-based detection per sector + variant. Flags values >3 IQR-distances from median.

In [8]:
outliers = findings[findings['rule'] == 'sector_outlier'].copy()
outliers['sector'] = outliers['detail'].str.extract(r'sector=([^,]+)')

sectors = ['All'] + sorted(outliers['sector'].dropna().unique())
columns = ['All'] + sorted(outliers['column'].dropna().unique())

sec_dd   = w.Dropdown(options=sectors, value='All', description='Sector:',
                      layout=w.Layout(width='240px'))
col_dd   = w.Dropdown(options=columns, value='All', description='Column:',
                      layout=w.Layout(width='240px'))
out_html = w.HTML()
_out_state = {'df': pd.DataFrame()}

_out_row, _out_note, _out_wl_btn, _out_chk_btn, _out_bl_btn, _out_msg = _exc_widgets(
    lambda i: _out_state['df'].iloc[i - 1] if 0 < i <= len(_out_state['df']) else None)

def _refresh_out(_=None):
    df = outliers.copy()
    if sec_dd.value != 'All': df = df[df['sector'] == sec_dd.value]
    if col_dd.value != 'All': df = df[df['column'] == col_dd.value]
    df = (
        df[['ticker','company_name','sector','period','report_date','column','value','rule','detail','edgar_url']]
        .sort_values('value', key=abs, ascending=False)
        .reset_index(drop=True)
    )
    _out_state['df'] = df
    _out_row.max = max(len(df), 1)
    df_show = _prep_df(df.copy())
    df_show.insert(0, '#', range(1, len(df_show) + 1))
    out_html.value = f'<p>{len(df):,} findings</p>' + _html(df_show).data

sec_dd.observe(_refresh_out, names='value')
col_dd.observe(_refresh_out, names='value')

display(w.HBox([sec_dd, col_dd]), out_html,
        w.HBox([_out_row, _out_note, _out_wl_btn, _out_chk_btn, _out_bl_btn]), _out_msg)
_refresh_out()

HTML(value='')

Output()

## 7. Sudden Jumps

Period-over-period changes >+500% or <−80% per ticker + variant.

In [9]:
jumps = findings[findings['rule'] == 'sudden_jump'].copy()
columns_j = ['All'] + sorted(jumps['column'].dropna().unique())

col_j     = w.Dropdown(options=columns_j, value='All', description='Column:',
                        layout=w.Layout(width='240px'))
jump_html = w.HTML()
_jmp_state = {'df': pd.DataFrame()}

_jmp_row, _jmp_note, _jmp_wl_btn, _jmp_chk_btn, _jmp_bl_btn, _jmp_msg = _exc_widgets(
    lambda i: _jmp_state['df'].iloc[i - 1] if 0 < i <= len(_jmp_state['df']) else None)

def _refresh_jump(_=None):
    df = jumps.copy()
    if col_j.value != 'All': df = df[df['column'] == col_j.value]
    df = (
        df[['ticker','company_name','period','report_date','column','value','rule','detail','edgar_url']]
        .sort_values('value', key=abs, ascending=False)
        .reset_index(drop=True)
    )
    _jmp_state['df'] = df
    _jmp_row.max = max(len(df), 1)
    df_show = _prep_df(df.copy())
    df_show.insert(0, '#', range(1, len(df_show) + 1))
    jump_html.value = f'<p>{len(df):,} findings</p>' + _html(df_show).data

col_j.observe(_refresh_jump, names='value')

display(col_j, jump_html,
        w.HBox([_jmp_row, _jmp_note, _jmp_wl_btn, _jmp_chk_btn, _jmp_bl_btn]), _jmp_msg)
_refresh_jump()

Dropdown(description='Column:', layout=Layout(width='240px'), options=('All', 'Net Income', 'Revenue', 'Total …

HTML(value='')

Output()

## 8. SEC Filings

Look up resolved filing URLs and errors per ticker.

In [10]:
sec = data['sec_filings'].copy()
sec_tickers = sorted(sec['ticker'].dropna().unique())

sec_cb = w.Combobox(
    options=sec_tickers,
    placeholder='Type a ticker…',
    description='Ticker:',
    ensure_option=False,
    layout=w.Layout(width='220px'),
)
out_s = w.Output()


def _show_sec(_=None):
    out_s.clear_output(wait=True)
    t = sec_cb.value.strip().upper()
    if not t:
        return
    rows = (
        sec[sec['ticker'] == t][['period', 'url', 'error']]
        .sort_values('period')
        .reset_index(drop=True)
    )
    with out_s:
        if rows.empty:
            print(f'No SEC filings for {t!r}')
        else:
            resolved = rows['url'].notna().sum()
            errors = rows['error'].notna().sum()
            print(f'{len(rows)} rows  |  {resolved} resolved  |  {errors} errors')
            display(_html(rows))


sec_cb.observe(_show_sec, names='value')
display(sec_cb, out_s)

Combobox(value='', description='Ticker:', layout=Layout(width='220px'), options=('A', 'A21', 'AA', 'AAC', 'AAC…

Output()

## 9. Exceptions

Current contents of `data/anomaly_whitelist.toml`.
Edit the file to add TOML comments (`#`) or remove entries.

In [11]:
from irp.anomalies.whitelist import load as load_whitelist

exc_out = w.Output()
reload_btn = w.Button(description='Reload', button_style='info')


def _show_exc(_=None):
    exc_out.clear_output(wait=True)
    df = load_whitelist()
    with exc_out:
        if df.empty:
            print('No exceptions yet.')
        else:
            print(f'{len(df)} exception(s)')
            display(_html(df))


reload_btn.on_click(_show_exc)
display(reload_btn, exc_out)
_show_exc()

Button(button_style='info', description='Reload', style=ButtonStyle())

Output()

## 10. Export

In [12]:
out_path = pathlib.Path('../data/data_quality/flagged_anomalies.csv')
findings.to_csv(out_path, index=False)
print(f'Exported {len(findings):,} findings → {out_path.resolve()}')

Exported 24,907 findings → /mnt/Dev/active_python_projects/investment_research_platform/data/data_quality/flagged_anomalies.csv
